# RealMLP Model — PharmShed
**Author:** Akhila Annireddy  
**Model:** RealMLP-TD (pytabkit)  
**Task:** Multi-class classification — predict which of 217 pharmaceuticals a person is prescribed based on demographics  
**Split strategy:** StratifiedGroupKFold (5-fold CV), grouped by Person_ID to prevent data leakage  
**Metrics:** Cohen's Kappa, MCC, macro/micro averaged accuracy, precision, recall, specificity, F2 score  

In [1]:
# Downloads and installs the two libraries we need that are not part of standard Python.
# pytabkit contains RealMLP and permetrics contains our evaluation metrics.
!pip install pytabkit permetrics xgboost

In [2]:
# Loads all the tools we need into memory.
# pandas for dataframes, numpy for math, LabelEncoder to convert drug names to integers,
# shuffle to randomize row order, RealMLP_TD_Classifier is our model,
# ClassificationMetric is for evaluation, StratifiedGroupKFold for our CV split.
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from pytabkit import RealMLP_TD_Classifier
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Reads both CSV files from the project folder into pandas dataframes.
# shape confirms row/column counts. columns check confirms names are exactly what we expect.
integrated_data = pd.read_csv('integrated_data.csv')
metadata = pd.read_csv('metadata.csv')

print("Integrated data shape:", integrated_data.shape)
print("Metadata shape:", metadata.shape)
print("\nIntegrated data columns:", integrated_data.columns.tolist())
print("Metadata columns:", metadata.columns.tolist())

Integrated data shape: (905728, 8)
Metadata shape: (905728, 6)

Integrated data columns: ['Unnamed: 0', 'Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity']
Metadata columns: ['Unnamed: 0', 'Observation_ID', 'Person_ID', 'NDC', 'Household_ID', 'Year']


In [4]:
# Removes the unwanted auto-generated index column from both dataframes.
integrated_data = integrated_data.drop(columns=['Unnamed: 0'])
metadata = metadata.drop(columns=['Unnamed: 0'])

print("Integrated data columns:", integrated_data.columns.tolist())
print("Metadata columns:", metadata.columns.tolist())
print("\nIntegrated data shape:", integrated_data.shape)
print("Metadata shape:", metadata.shape)

Integrated data columns: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity']
Metadata columns: ['Observation_ID', 'Person_ID', 'NDC', 'Household_ID', 'Year']

Integrated data shape: (905728, 7)
Metadata shape: (905728, 5)


In [5]:
# Pulls only Person_ID from metadata and attaches it to integrated_data using Observation_ID as the key.
# Person_ID is needed solely for StratifiedGroupKFold grouping — it will not be used as a model feature.
person_id_map = metadata[['Observation_ID', 'Person_ID']]
integrated_data = integrated_data.merge(person_id_map, on='Observation_ID', how='left')

print("Integrated data columns after join:", integrated_data.columns.tolist())
print("Integrated data shape after join:", integrated_data.shape)
print("\nMissing Person_IDs:", integrated_data['Person_ID'].isnull().sum())

Integrated data columns after join: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Person_ID']
Integrated data shape after join: (905728, 8)

Missing Person_IDs: 0


In [6]:
# EDA: understand the data before splitting.
# Unique person count, unique drugs, drug distribution (imbalance check), missing values.
print("Unique persons:", integrated_data['Person_ID'].nunique())
print("Unique drugs:", integrated_data['Drug'].nunique())

print("\nTop 10 most prescribed drugs:")
print(integrated_data['Drug'].value_counts().head(10))

print("\nBottom 5 rarest drugs:")
print(integrated_data['Drug'].value_counts().tail(5))

print("\nMissing values per column:")
print(integrated_data.isnull().sum())

Unique persons: 126967
Unique drugs: 217

Top 10 most prescribed drugs:
Drug
no prescriptions    97497
atorvastatin        38557
lisinopril          35851
metformin           33777
amlodipine          28135
metoprolol          25001
albuterol           23188
omeprazole          22770
losartan            18670
gabapentin          18337
Name: count, dtype: int64

Bottom 5 rarest drugs:
Drug
sulfamethoxazole    72
trimethoprim        72
gentamicin          64
piroxicam           61
ivermectin          29
Name: count, dtype: int64

Missing values per column:
Observation_ID        0
Drug                  0
Age                   0
Sex                   0
Family_income         0
Insurance_coverage    0
Race_ethnicity        0
Person_ID             0
dtype: int64


## Data Splitting — StratifiedGroupKFold (5-fold CV)

**Why StratifiedGroupKFold?**
- **Grouped** by `Person_ID`: all rows for the same person (including their refills across years) always stay together in the same fold. This prevents data leakage where the model sees a person's refills in training and then predicts their other refills in validation.
- **Stratified** by `Drug`: ensures each fold has approximately the same drug class distribution as the full dataset. This is critical given our severe class imbalance (atorvastatin: 38,557 rows vs ivermectin: 29 rows).
- **5-fold CV**: gives us 5 independent train/validation splits so we can average performance metrics and report variance, which is more rigorous than a single 80/20 split.

**Final model**: after CV we train a final model on the full dataset (all folds combined) for use in the ensemble.

In [7]:
# Fit LabelEncoder once on the full dataset BEFORE the CV loop.
# This ensures consistent integer-to-drug mapping across all folds.
# Fitting only on train inside the loop would cause different folds to have different encodings.

feature_cols = ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity']
categorical_cols = ['Sex', 'Insurance_coverage', 'Race_ethnicity']
numeric_cols = ['Age', 'Family_income']
target_col = 'Drug'

le = LabelEncoder()
integrated_data['Drug_encoded'] = le.fit_transform(integrated_data[target_col])

print("Unique classes in encoder:", len(le.classes_))
print("Sample mapping (first 5):")
for i, drug in enumerate(le.classes_[:5]):
    print(f"  {drug} -> {i}")

Unique classes in encoder: 217
Sample mapping (first 5):
  acetaminophen -> 0
  acyclovir -> 1
  adapalene -> 2
  albuterol -> 3
  alendronate -> 4


In [8]:
# StratifiedGroupKFold 5-fold cross-validation.
#
# For each fold:
#   1. Split indices by Person_ID groups and stratify by Drug label
#   2. Shuffle rows within train and validation sets (so refills aren't bunched)
#   3. Set categorical dtypes so RealMLP handles encoding internally
#   4. Train RealMLP-TD on train fold
#   5. Predict on validation fold
#   6. Compute and store all required metrics
#
# NOTE: RealMLP-TD handles its own numeric scaling and categorical encoding internally.
# We do NOT scale numerics or one-hot encode categoricals ourselves — doing so would conflict
# with RealMLP's built-in preprocessing and could cause data leakage.

sgkf = StratifiedGroupKFold(n_splits=5)

X = integrated_data[feature_cols].copy()
y = integrated_data['Drug_encoded'].values
groups = integrated_data['Person_ID'].values

# Storage for metrics across folds
fold_results = []

# Storage for per-drug recall across folds (needed for ensemble model selection)
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f"\n{'='*50}")
    print(f"FOLD {fold_num}/5")
    print(f"{'='*50}")
    
    # --- Split ---
    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold   = X.iloc[val_idx].copy()
    y_train_fold = y[train_idx]
    y_val_fold   = y[val_idx]
    
    # --- Shuffle rows within each fold (so refills aren't bunched together) ---
    train_order = np.random.RandomState(42).permutation(len(X_train_fold))
    val_order   = np.random.RandomState(42).permutation(len(X_val_fold))
    X_train_fold = X_train_fold.iloc[train_order].reset_index(drop=True)
    y_train_fold = y_train_fold[train_order]
    X_val_fold   = X_val_fold.iloc[val_order].reset_index(drop=True)
    y_val_fold   = y_val_fold[val_order]
    
    # --- Set categorical dtypes so RealMLP uses its built-in categorical encoding ---
    for col in categorical_cols:
        X_train_fold[col] = X_train_fold[col].astype('category')
        X_val_fold[col]   = X_val_fold[col].astype('category')
    
    print(f"Train size: {len(X_train_fold):,} rows | Val size: {len(X_val_fold):,} rows")
    print(f"Unique drugs in train: {len(np.unique(y_train_fold))} | in val: {len(np.unique(y_val_fold))}")
    
    # --- Train RealMLP-TD ---
    # TD version uses tuned default parameters — no manual hyperparameter search needed.
    # device='mps' uses Apple Silicon GPU for faster training.
    model = RealMLP_TD_Classifier(
        device='mps',
        random_state=42,
        n_epochs=30,
    )
    model.fit(X_train_fold, y_train_fold)
    print(f"Fold {fold_num} training complete.")
    
    # --- Predict ---
    y_pred_fold = model.predict(X_val_fold)
    
    # --- Metrics ---
    # Overall accuracy
    acc = accuracy_score(y_val_fold, y_pred_fold)
    
    # Cohen's Kappa: measures agreement between predictions and ground truth,
    # accounting for chance. Better than raw accuracy for imbalanced classes.
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    
    # Matthews Correlation Coefficient: symmetric metric that handles class imbalance well.
    # Ranges from -1 (worst) to +1 (perfect). 0 = random.
    mcc = matthews_corrcoef(y_val_fold, y_pred_fold)
    
    # Macro and micro averaged precision, recall, F1, F2 using permetrics
    # Macro: compute metric per class then average equally — treats rare and common drugs equally.
    # Micro: aggregate counts across all classes then compute — dominated by common drugs.
    # We report both so we can see both perspectives.
    evaluator = ClassificationMetric(y_val_fold, y_pred_fold)
    
    macro_precision  = evaluator.precision_score(average='macro')
    micro_precision  = evaluator.precision_score(average='micro')
    macro_recall     = evaluator.recall_score(average='macro')
    micro_recall     = evaluator.recall_score(average='micro')
    macro_f1         = evaluator.f1_score(average='macro')
    micro_f1         = evaluator.f1_score(average='micro')
    
    # F2 score: weights recall twice as much as precision.
    # More important for us because false negatives (missing a drug) = more pollution.
    macro_f2 = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2 = evaluator.fbeta_score(beta=2, average='micro')
    
    # Store fold-level summary
    fold_results.append({
        'fold': fold_num,
        'accuracy': acc,
        'cohen_kappa': kappa,
        'mcc': mcc,
        'macro_precision': macro_precision,
        'micro_precision': micro_precision,
        'macro_recall': macro_recall,
        'micro_recall': micro_recall,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'macro_f2': macro_f2,
        'micro_f2': micro_f2,
    })
    
    # Per-drug recall for this fold (for ensemble model selection)
    # classification_report gives us per-class precision/recall/f1
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)
    
    print(f"Fold {fold_num} results:")
    print(f"  Accuracy:        {acc:.4f}")
    print(f"  Cohen Kappa:     {kappa:.4f}")
    print(f"  MCC:             {mcc:.4f}")
    print(f"  Macro Recall:    {macro_recall:.4f}")
    print(f"  Micro Recall:    {micro_recall:.4f}")
    print(f"  Macro F2:        {macro_f2:.4f}")

print("\n" + "="*50)
print("ALL FOLDS COMPLETE")
print("="*50)


FOLD 1/5
Train size: 724,582 rows | Val size: 181,146 rows
Unique drugs in train: 217 | in val: 217


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=30` reached.


Fold 1 training complete.


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Fold 1 results:
  Accuracy:        0.1296
  Cohen Kappa:     0.0746
  MCC:             0.0796
  Macro Recall:    0.0099
  Micro Recall:    0.1296
  Macro F2:        0.0073

FOLD 2/5
Train size: 724,582 rows | Val size: 181,146 rows
Unique drugs in train: 217 | in val: 217


...

Fold 4 results:
  Accuracy:        0.1248
  Cohen Kappa:     0.0684
  MCC:             0.0733
  Macro Recall:    0.0092
  Micro Recall:    0.1248
  Macro F2:        0.0068

FOLD 5/5
Train size: 724,583 rows | Val size: 181,145 rows
Unique drugs in train: 217 | in val: 217


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=30` reached.


Fold 5 training complete.


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Fold 5 results:
  Accuracy:        0.1268
  Cohen Kappa:     0.0703
  MCC:             0.0754
  Macro Recall:    0.0092
  Micro Recall:    0.1268
  Macro F2:        0.0067

ALL FOLDS COMPLETE


In [9]:
# Summarize CV results across all 5 folds.
# Report mean and standard deviation for each metric.
# High std = unstable model; low std = consistent performance across folds.

results_df = pd.DataFrame(fold_results)
print("Per-fold results:")
print(results_df.to_string(index=False))

print("\nMean ± Std across 5 folds:")
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f"  {col:25s}: {mean:.4f} ± {std:.4f}")

# Save fold summary to CSV for lab notebook / paper
results_df.to_csv('realmlp_cv_results.csv', index=False)
print("\nCV results saved to realmlp_cv_results.csv")

Per-fold results:
 fold  accuracy  cohen_kappa      mcc  macro_precision  micro_precision  macro_recall  micro_recall  macro_f1  micro_f1  macro_f2  micro_f2
    1  0.129592     0.074557 0.079621         0.012592         0.129592      0.009857      0.129592  0.005993  0.129592  0.007343  0.129592
    2  0.126826     0.070931 0.076131         0.007878         0.126826      0.009308      0.126826  0.005427  0.126826  0.006828  0.126826
    3  0.126489     0.069237 0.074654         0.006692         0.126489      0.009031      0.126489  0.005218  0.126489  0.006579  0.126489
    4  0.124751     0.068442 0.073303         0.006410         0.124751      0.009181      0.124751  0.005401  0.124751  0.006778  0.124751
    5  0.126799     0.070318 0.075439         0.006144         0.126799      0.009230      0.126799  0.005196  0.126799  0.006705  0.126799

Mean ± Std across 5 folds:
  accuracy                 : 0.1269 ± 0.0017
  cohen_kappa              : 0.0707 ± 0.0024
  mcc                   

In [10]:
# Compute average per-drug recall across all 5 folds.
# This is the key output for ensemble model selection:

per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols = [c for c in per_drug_df.columns if c != 'fold']

# Average recall per drug across folds
mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_RealMLP']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_RealMLP', ascending=False)

print("Top 20 drugs by mean recall (RealMLP performs best here):")
print(mean_drug_recall.head(20).to_string(index=False))

print("\nBottom 20 drugs by mean recall (RealMLP struggles here):")
print(mean_drug_recall.tail(20).to_string(index=False))

# Save per-drug recall for ensemble comparison
mean_drug_recall.to_csv('realmlp_per_drug_recall.csv', index=False)
print("\nPer-drug recall saved to realmlp_per_drug_recall.csv")

Top 20 drugs by mean recall (RealMLP performs best here):
               Drug  Mean_Recall_RealMLP
   no prescriptions             0.825974
       atorvastatin             0.396892
         amlodipine             0.161187
          metformin             0.123072
          albuterol             0.107770
         lisinopril             0.089928
         metoprolol             0.089357
         tamsulosin             0.048234
         gabapentin             0.043901
    methylphenidate             0.042886
           losartan             0.016979
         omeprazole             0.014493
          memantine             0.010370
  divalproex sodium             0.009600
hydrochlorothiazide             0.007951
       aripiprazole             0.005200
           warfarin             0.004850
        lamotrigine             0.003802
         furosemide             0.003603
        clopidogrel             0.003152

Bottom 20 drugs by mean recall (RealMLP struggles here):
             Drug  Mean

## Final Model — Train on Full Dataset

After CV gives us confidence in the model's performance, we train one final model on the **entire dataset** (all years 2014–2021). This is the model that will be used in the ensemble and for internal validation on MEPS 2022 data.

The test set for final evaluation is **MEPS 2022** (held out entirely from training — this is the internal validation set per Ren's framework).

In [11]:
# Train final RealMLP model on the full integrated dataset (2014-2021).
# MEPS 2022 data is held out for internal validation (separate step).
#
# IMPORTANT: categorical dtypes must be set before feeding to model.
# RealMLP handles all scaling and encoding internally — do not preprocess further.

X_final = integrated_data[feature_cols].copy()
y_final = integrated_data['Drug_encoded'].values

for col in categorical_cols:
    X_final[col] = X_final[col].astype('category')

# Shuffle before training
X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print("Final model training data size:", X_final.shape)
print("Number of classes:", len(le.classes_))
print("\nTraining final RealMLP model...")

final_model = RealMLP_TD_Classifier(
    device='mps',
    random_state=42,
    n_epochs=30,
)
final_model.fit(X_final, y_final)
print("Final model training complete!")

Final model training data size: (905728, 5)
Number of classes: 217

Training final RealMLP model...


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

In [ ]:
# Internal validation on MEPS 2022 data (held-out test set).
# Load the 2022 data, apply the same preprocessing, and evaluate.
# This follows Ren's train/validate/test framework where 2022 is the test set.

# Load 2022 data
data_2022 = pd.read_csv('data_2022.csv')
print("2022 data shape:", data_2022.shape)
print("Columns:", data_2022.columns.tolist())

# Drop unnamed index if present
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

# Check which drugs in 2022 are in our label encoder (trained on 2014-2021)
known_drugs = set(le.classes_)
drugs_2022  = set(data_2022['Drug'].unique())
unseen_drugs = drugs_2022 - known_drugs
print(f"\nDrugs in 2022 data: {len(drugs_2022)}")
print(f"Drugs known to model: {len(known_drugs)}")
print(f"Unseen drugs (will be dropped): {len(unseen_drugs)}")
if unseen_drugs:
    print("Unseen drug names:", unseen_drugs)

# Keep only rows with drugs the model knows
data_2022_filtered = data_2022[data_2022['Drug'].isin(known_drugs)].copy()
print(f"\n2022 rows after filtering unseen drugs: {len(data_2022_filtered):,}")

# Prepare features and target
X_2022 = data_2022_filtered[feature_cols].copy()
for col in categorical_cols:
    X_2022[col] = X_2022[col].astype('category')

y_2022_encoded = le.transform(data_2022_filtered['Drug'])

# Predict
y_pred_2022 = final_model.predict(X_2022)

# Compute metrics
acc_2022   = accuracy_score(y_2022_encoded, y_pred_2022)
kappa_2022 = cohen_kappa_score(y_2022_encoded, y_pred_2022)
mcc_2022   = matthews_corrcoef(y_2022_encoded, y_pred_2022)

evaluator_2022 = ClassificationMetric(y_2022_encoded, y_pred_2022)
macro_recall_2022 = evaluator_2022.recall_score(average='macro')
micro_recall_2022 = evaluator_2022.recall_score(average='micro')
macro_f2_2022     = evaluator_2022.fbeta_score(beta=2, average='macro')
micro_f2_2022     = evaluator_2022.fbeta_score(beta=2, average='micro')
macro_prec_2022   = evaluator_2022.precision_score(average='macro')
micro_prec_2022   = evaluator_2022.precision_score(average='micro')

print("\n" + "="*50)
print("INTERNAL VALIDATION — MEPS 2022 Results")
print("="*50)
print(f"Accuracy:          {acc_2022:.4f}")
print(f"Cohen Kappa:       {kappa_2022:.4f}")
print(f"MCC:               {mcc_2022:.4f}")
print(f"Macro Precision:   {macro_prec_2022:.4f}")
print(f"Micro Precision:   {micro_prec_2022:.4f}")
print(f"Macro Recall:      {macro_recall_2022:.4f}")
print(f"Micro Recall:      {micro_recall_2022:.4f}")
print(f"Macro F2:          {macro_f2_2022:.4f}")
print(f"Micro F2:          {micro_f2_2022:.4f}")

# Per-drug recall on 2022 data
report_2022 = classification_report(
    y_2022_encoded, y_pred_2022,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)
drug_recall_2022 = pd.DataFrame([
    {'Drug': drug, 'Recall_2022': report_2022[drug]['recall'],
     'Precision_2022': report_2022[drug]['precision'],
     'F1_2022': report_2022[drug]['f1-score'],
     'Support_2022': report_2022[drug]['support']}
    for drug in le.classes_ if drug in report_2022
]).sort_values('Recall_2022', ascending=False)

print("\nTop 15 drugs by recall on 2022 data:")
print(drug_recall_2022.head(15).to_string(index=False))

print("\nBottom 15 drugs by recall on 2022 data:")
print(drug_recall_2022.tail(15).to_string(index=False))

# Save 2022 validation results
drug_recall_2022.to_csv('realmlp_2022_per_drug_metrics.csv', index=False)
print("\n2022 per-drug metrics saved to realmlp_2022_per_drug_metrics.csv")

# Save summary validation metrics to CSV
validation_summary = pd.DataFrame([{
    'model': 'RealMLP',
    'dataset': 'MEPS_2022_internal_validation',
    'accuracy': acc_2022,
    'cohen_kappa': kappa_2022,
    'mcc': mcc_2022,
    'macro_precision': macro_prec_2022,
    'micro_precision': micro_prec_2022,
    'macro_recall': macro_recall_2022,
    'micro_recall': micro_recall_2022,
    'macro_f2': macro_f2_2022,
    'micro_f2': micro_f2_2022,
}])
validation_summary.to_csv('realmlp_validation_summary.csv', index=False)
print("Validation summary saved to realmlp_validation_summary.csv")

In [ ]:
# Inspect prediction distribution on 2022 data.
# Compares how many times each drug was predicted vs how many times it actually appeared.
# This helps diagnose whether the class imbalance issue (over-predicting common drugs) persists.

pred_drugs_2022   = le.inverse_transform(y_pred_2022)
actual_drugs_2022 = le.inverse_transform(y_2022_encoded)

pred_counts   = pd.Series(pred_drugs_2022).value_counts().rename('predicted')
actual_counts = pd.Series(actual_drugs_2022).value_counts().rename('actual')

dist_compare = pd.concat([actual_counts, pred_counts], axis=1).fillna(0).astype(int)
dist_compare['ratio_pred_to_actual'] = (dist_compare['predicted'] / dist_compare['actual'].replace(0, 1)).round(2)
dist_compare = dist_compare.sort_values('actual', ascending=False)

print("Prediction vs actual distribution (top 20 most common drugs):")
print(dist_compare.head(20))

print("\nPrediction vs actual distribution (bottom 20 rarest drugs):")
print(dist_compare.tail(20))

# ratio > 1 means model over-predicts this drug
# ratio < 1 means model under-predicts this drug
# ratio = 0 means model never predicts this drug
never_predicted = dist_compare[dist_compare['predicted'] == 0]
print(f"\nDrugs never predicted by the model: {len(never_predicted)}")
if len(never_predicted) > 0:
    print(never_predicted.index.tolist())

## Summary of Outputs

| File | Contents |
|------|----------|
| `realmlp_cv_results.csv` | Mean ± std for all metrics across 5 CV folds |
| `realmlp_per_drug_recall.csv` | Average recall per drug across 5 folds (for ensemble selection) |
| `realmlp_2022_per_drug_metrics.csv` | Per-drug recall, precision, F1 on MEPS 2022 internal validation |
| `realmlp_validation_summary.csv` | Overall validation metrics on MEPS 2022 |
